In [ ]:


# %pip install pygeoprocessing taskgraph pyogrio

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

import argparse
import logging
import math
import os
import sys
import tempfile

import geopandas as gpd
import logging
import numpy
from osgeo import gdal, ogr, osr
import pandas as pd
import pygeoprocessing
import rasterio
import taskgraph

Note: you may need to restart the kernel to use updated packages.


In [ ]:
BASE_DIR      = Path(r"C:\Users\yingjiel\Documents\mongolia-mining")
ES_TABLE_PATH = BASE_DIR / 'data' / 'natcap_footprint_data' / 'ecosystem_service_layer_table.csv'
DATASETS      = ["Maus2022"]   # add "Maus2022" to run both
N_WORKERS     = 0              # 0 = serial (safe); -1 = all CPU cores
OUTPUT_FORMATS = ['gpkg', 'csv']


# ---------------------------
# Input files
# ---------------------------

mining_fp   = BASE_DIR / 'data' / 'raw/_mining-viz/Tang_v2_7894216' / "74548_projected polygons.shp";              data_label = "Tang2023"
mining_fp   = BASE_DIR / 'data' / 'raw/_mining-viz/Maus-etal_2022_V2_allfiles' / "global_mining_polygons_v2.gpkg"; data_label = "Maus2022"
mongolia_fp = BASE_DIR / "ne_50m_admin_0_countries_MNG.shp"

TypeError: unsupported operand type(s) for /: 'builtin_function_or_method' and 'str'

## footprint

### function 

In [ ]:

def footprint_stats(footprint_path, es_table_path, id_col='es_id', n_workers=-1):
    """Calculate and record stats of ecosystem service values under footprints.

    Args:
        footprint_path (str): path to a GDAL-supported footprint polygon vector
        es_table_path (str): path to the ecosystem service CSV, which should
            have the following columns: es_id (the unique identifier for
            each ecosystem service); path (the path to the global ecosystem
            service raster); and the numbers 0 through 100, storing the
            integer percentile values for each ecosystem service globally.
        out_path (str): path to write out the resulting footprint stats vector

    Returns:
        None
    """
    graph = taskgraph.TaskGraph(os.getcwd(), n_workers=n_workers)

    logger.info('calculating statistics under footprints...')
    footprint_gdf = gpd.read_file(
        footprint_path, engine='pyogrio', fid_as_index=True)

    if not ((footprint_gdf.geom_type == 'Polygon') |
            (footprint_gdf.geom_type == 'MultiPolygon')).all():
        raise ValueError('All geometries in the asset vector must be polygons or multipolygons')

    es_df = pd.read_csv(es_table_path)
    es_id_to_task = {}
    for i, row in es_df.iterrows():
        es_id = row[id_col]
        path = os.path.abspath(os.path.join(
            os.path.dirname(es_table_path), row['es_value_path']))
        pixel_size = pygeoprocessing.get_raster_info(path)['pixel_size']
        es_df.loc[i, 'pixel_area'] = abs(pixel_size[0] * pixel_size[1])

        es_id_to_task[es_id] = graph.add_task(
            func=pygeoprocessing.zonal_statistics,
            args=((path, 1), footprint_path),
            target_path_list=[],
            task_name=f'{es_id} stats',
            store_result=True)

    graph.close()
    graph.join()

    for _, row in es_df.iterrows():
        es_id = row[id_col]
        zonal_stats = es_id_to_task[es_id].get()
        for stat in ['max', 'sum', 'count', 'nodata_count']:
            footprint_gdf[f'{es_id}_{stat}'] = pd.Series(
                footprint_gdf.index.to_series().map(
                    lambda fid: zonal_stats[fid][stat]))

        # use the sum to calculate the mean, but leave it out of the final result
        footprint_gdf.loc[footprint_gdf[f'{es_id}_count'] > 0, f'{es_id}_mean'] = (
            footprint_gdf[f'{es_id}_sum'] / footprint_gdf[f'{es_id}_count'])
        footprint_gdf = footprint_gdf.drop(f'{es_id}_sum', axis=1)

        # flag assets that have an ES value greater than the threshold
        footprint_gdf[f'{es_id}_flag'] = (
            footprint_gdf[f'{es_id}_max'] > row['flag_threshold'])

        # calculate the area-adjusted sum, which should be interpreted as an index
        footprint_gdf[f'{es_id}_adj_sum'] = (
            footprint_gdf[f'{es_id}_mean'] * footprint_gdf.area / row['pixel_area'])

    return footprint_gdf